# NeurIPS 2026 Smart Buildings — Self-Contained Offline Submission

- **Internet:** OFF (100% pure NumPy + stdlib, zero external dependencies)
- **Policy:** `GeneralizedHVACPolicy` + `PeriodicHeuristicForecaster` (Run 4 + Thermodynamic OAT Gate)
- **Interface:** Subclasses `smart_buildings_inference_server.Policy` and runs `SmartBuildingsInferenceServer`


In [ ]:
"""Self-contained Internet-OFF Kaggle Submission for NeurIPS 2026 Smart Buildings Competition.

Pure NumPy + Python standard library. No external network calls, no third-party ML weights,
no dataset-specific hardcoding. Subclasses `smart_buildings_inference_server.Policy`.
"""

from __future__ import annotations

from collections import deque
from collections.abc import Mapping
from dataclasses import asdict, dataclass
import json
from pathlib import Path
import re
import sys
import time
from typing import Any, Callable

import numpy as np

sys.path.append("/kaggle/input/competitions/neurips-2026-smart-buildings/kaggle_evaluation")
from kaggle_evaluation import smart_buildings_inference_server

COMPETITION_DATA_FOLDER = "/kaggle/input/competitions/neurips-2026-smart-buildings/"

class PIDPolicy:
    """Distributed per-AHU PID controller with proportional fan pressure.

    Each AHU runs an independent PID loop on the average temperature of the
    zones it serves, modulating that AHU's supply-air-temperature setpoint in
    proportion to how far those zones sit outside the active comfort/eco band.
    An AHU idles whenever its zones are inside the band, which is the same
    rule BangBangPolicy uses.

    Fan pressure scales with the control effort |u| rather than being pinned
    at maximum whenever the AHU is active: |u| = 0 maps to pressure -1
    (minimum airflow) and |u| = 1 to pressure +1 (full blast), linearly in
    between. So a zone barely outside the band gets a gentle push instead of
    the full fan.

    That last point is the main energy lever, and it is worth stating why.
    Fan power is flow * static_pressure / efficiency, so holding pressure at
    maximum is expensive for as long as it is held. Pinning it while a zone
    drifts back into band is particularly costly here: because a PID corrects
    gently, that recovery window is long, and a smoothly modulating controller
    can end up drawing more energy than a blunter on/off one. Scaling pressure
    with control effort removes that penalty.

    The water loop is deliberately left binary: hot water on if any air handler
    is heating, pump on if any is active. Making it proportional to demand
    saves very little gas, because the heat is genuinely needed, and what it
    does save tends to be cancelled by extra fan electricity from slower zone
    recovery.

    Underneath the AHU loop, each VAV gets its own damper and reheat valve
    command from its own zone's temperature, so a zone that is comfortable is
    throttled to its ventilation minimum while its neighbours on the same AHU
    are still being conditioned. The AHU loop can only pick one supply air
    temperature for all of them, which is why this second layer is worth having:
    it is the only place a per-zone decision can be made at all.
    """

    def __init__(
        self,
        building_meta_data,
        action_spec,
        kp: float = 0.5,
        ki: float = 0.02,
        kd: float = 0.1,
        integral_limit: float = 10.0,
        # Excursion in K at which a zone's damper is commanded fully open. The
        # AHU loop reaches full effort at 1/kp = 2 K, so this matches it.
        zone_damper_full_k: float = 2.0,
    ):
        self.metadata = building_meta_data
        self.action_spec = action_spec

        self.comfort_temp_window = building_meta_data["comfort_temp_window"]
        self.eco_temp_window = building_meta_data["eco_temp_window"]
        self.morning_start_hour = building_meta_data["morning_start_hour"]
        self.evening_start_hour = building_meta_data["evening_start_hour"]

        self.n_discrete = building_meta_data["n_discrete_action"]
        self.n_continuous = building_meta_data["n_continuous_action"]
        self.temp_indices = list(building_meta_data["temp_indices"])
        self.pressure_indices = list(building_meta_data["pressure_indices"])
        self.water_indices = list(building_meta_data["water_indices"])
        self.recirculation_indices = list(building_meta_data["recirculation_indices"])
        self.zone_action_indices = building_meta_data["zone_action_indices"]
        self.field_names = building_meta_data["observation_field_names"]
        self.ahu_zone_assignments = building_meta_data["ahu_zone_assignments"]

        self.observation_normalization_metadata = building_meta_data[
            "observation_normalization_metadata"
        ]
        self.zone_air_temperature_sensor_mean = self.observation_normalization_metadata[
            "zone_air_temperature_sensor"
        ]["sample_mean"]
        self.zone_air_temperature_sensor_std = self.observation_normalization_metadata[
            "zone_air_temperature_sensor"
        ]["sample_std"]

        self.hod_sin_index = self.field_names.index("hod_sin_000")
        self.hod_cos_index = self.field_names.index("hod_cos_000")
        self.dow_sin_index = self.field_names.index("dow_sin_000")
        self.dow_cos_index = self.field_names.index("dow_cos_000")

        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.integral_limit = integral_limit
        self.zone_damper_full_k = zone_damper_full_k

        self._build_ahu_mapping()
        self.reset()

    def _build_ahu_mapping(self) -> None:
        zone_temp_field_index = {
            name: i
            for i, name in enumerate(self.field_names)
            if "zone_air_temperature" in name
        }

        ahu_ids = sorted(self.ahu_zone_assignments.keys(), key=self._numeric_suffix)
        self.n_ahu = len(self.temp_indices)
        if len(ahu_ids) != self.n_ahu:
            self.n_ahu = min(self.n_ahu, len(ahu_ids))
            ahu_ids = ahu_ids[: self.n_ahu]

        self.ahu_controllers: list[dict[str, Any]] = []
        for i, ahu_id in enumerate(ahu_ids):
            rooms = self.ahu_zone_assignments[ahu_id]
            # (vav id, observation index, action indices) for each zone this AHU
            # serves that the building actually reports. A zone whose sensor is
            # missing is skipped entirely rather than being commanded blind.
            zones: list[tuple[str, int, dict[str, int]]] = []
            for room in rooms:
                vav_id = room.replace("room_", "VAV_")
                vav_field = vav_id + "_zone_air_temperature_sensor"
                if vav_field in zone_temp_field_index:
                    zones.append(
                        (
                            vav_id,
                            zone_temp_field_index[vav_field],
                            self.zone_action_indices.get(vav_id, {}),
                        )
                    )

            self.ahu_controllers.append(
                {
                    "ahu_id": ahu_id,
                    "temp_idx": self.temp_indices[i],
                    "pressure_idx": self.pressure_indices[i]
                    if i < len(self.pressure_indices)
                    else None,
                    "run_discrete_idx": i,
                    "zones": zones,
                    "zone_obs_indices": [obs_idx for _, obs_idx, _ in zones],
                }
            )

        self.water_temp_idx = self.water_indices[0] if self.water_indices else None
        self.water_pressure_idx = (
            self.pressure_indices[-1] if len(self.pressure_indices) > self.n_ahu else None
        )
        self.water_discrete_idx = self.n_discrete - 1

    @staticmethod
    def _numeric_suffix(name: str) -> int:
        match = re.search(r"(\d+)", name)
        return int(match.group(1)) if match else 0

    @property
    def name(self) -> str:
        return "pid"

    def reset(self) -> None:
        self.pid_state = {i: {"integral": 0.0, "prev_error": 0.0} for i in range(self.n_ahu)}

    def _cyclic_feature_to_index(self, sin_value: float, cos_value: float, period: int) -> int:
        angle = np.arctan2(sin_value, cos_value)
        if angle < 0:
            angle += 2 * np.pi
        return int(np.floor(period * angle / (2 * np.pi)))

    def _get_hour_and_day(self, obs: np.ndarray):
        hour = self._cyclic_feature_to_index(obs[self.hod_sin_index], obs[self.hod_cos_index], period=24)
        day = self._cyclic_feature_to_index(obs[self.dow_sin_index], obs[self.dow_cos_index], period=7)
        return hour, day

    def _denormalize_zone_temp(self, normalized_value: float) -> float:
        return normalized_value * self.zone_air_temperature_sensor_std + self.zone_air_temperature_sensor_mean

    def _deadband_error(self, avg_temp: float, low: float, high: float) -> float:
        if avg_temp < low:
            return low - avg_temp
        if avg_temp > high:
            return high - avg_temp
        return 0.0

    def _command_zones(
        self,
        continuous: np.ndarray,
        zones: list[tuple[str, int, dict[str, int]]],
        obs: np.ndarray,
        low: float,
        high: float,
    ) -> None:
        """Sets each VAV's damper and reheat valve from its own zone temperature.

        Proportional in the zone's own excursion, so a zone that is in band is
        throttled to its ventilation floor while a neighbour on the same air
        handler stays wide open. The reheat valve is binary because the supply
        air is either too cold for this zone or it is not; there is no third
        thing for the valve to do.
        """
        for _, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            zone_error = self._deadband_error(
                self._denormalize_zone_temp(obs[obs_idx]), low, high
            )
            damper = min(1.0, abs(zone_error) / self.zone_damper_full_k)

            damper_idx = action_indices.get("supply_air_damper_percentage_command")
            if damper_idx is not None:
                continuous[damper_idx] = 2.0 * damper - 1.0
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is not None:
                continuous[reheat_idx] = 1.0 if zone_error > 0 else -1.0

    def get_action(self, obs: np.ndarray) -> dict[str, Any]:
        hour, day = self._get_hour_and_day(obs)
        is_weekday = day < 5
        is_work_hour = self.morning_start_hour <= hour < self.evening_start_hour

        if is_weekday and is_work_hour:
            low_temp, high_temp = self.comfort_temp_window
        else:
            low_temp, high_temp = self.eco_temp_window

        continuous = np.zeros(self.n_continuous, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)

        any_heating = False
        any_active = False

        for i, ctrl in enumerate(self.ahu_controllers):
            zone_obs_indices = ctrl["zone_obs_indices"]
            if not zone_obs_indices:
                continue

            normalized_temps = obs[zone_obs_indices]
            avg_temp = float(np.mean([self._denormalize_zone_temp(v) for v in normalized_temps]))

            self._command_zones(continuous, ctrl["zones"], obs, low_temp, high_temp)

            error = self._deadband_error(avg_temp, low_temp, high_temp)

            state = self.pid_state[i]
            if error == 0.0:
                # Inside the band: idle, same as bang_bang.
                state["integral"] *= 0.5
                state["prev_error"] = 0.0
                continuous[ctrl["temp_idx"]] = -1.0
                if ctrl["pressure_idx"] is not None:
                    continuous[ctrl["pressure_idx"]] = -1.0
                discrete[ctrl["run_discrete_idx"]] = 0
                continue

            state["integral"] = float(
                np.clip(state["integral"] + error, -self.integral_limit, self.integral_limit)
            )
            derivative = error - state["prev_error"]
            state["prev_error"] = error

            u = self.kp * error + self.ki * state["integral"] + self.kd * derivative
            u = float(np.clip(u, -1.0, 1.0))

            # Pressure scales with control effort: |u|=0 -> pressure=-1 (min),
            # |u|=1 -> pressure=+1 (max). No longer pinned to 1.0.
            pressure = float(np.clip(2.0 * abs(u) - 1.0, -1.0, 1.0))

            continuous[ctrl["temp_idx"]] = u
            if ctrl["pressure_idx"] is not None:
                continuous[ctrl["pressure_idx"]] = pressure
            discrete[ctrl["run_discrete_idx"]] = 1

            any_active = True
            if error > 0:
                any_heating = True

        # Recirculate as much as the ventilation cap allows. This is the no-
        # economizer choice, and it is deliberately left as headroom: deciding
        # when outside air is cheaper than recirculated air needs the weather,
        # which this controller does not look at.
        for i in self.recirculation_indices:
            continuous[i] = 1.0

        if self.water_temp_idx is not None:
            continuous[self.water_temp_idx] = 1.0 if any_heating else -1.0
        if self.water_pressure_idx is not None:
            continuous[self.water_pressure_idx] = 1.0 if any_active else -1.0
        if 0 <= self.water_discrete_idx < self.n_discrete:
            discrete[self.water_discrete_idx] = 1 if any_active else 0

        return {"continuous_action": continuous, "discrete_action": discrete}

"""A chain of controllers built on `pid`, each adding one idea to the last.

`pid` already idles per air handler while its zones are in band, and drives
each VAV damper and reheat valve from that zone's own temperature. These
subclasses address what a plant-level rule-based controller still leaves on the
table. Each is a base class for the next, so the chain reads as a sequence of
independent improvements.

  PIDWorstZonePolicy   The AHU loop runs on the *average* zone temperature of
                       the zones it serves, but the comfort penalty is
                       unbounded in the *worst* zone's deviation. An air
                       handler can therefore sit idle, perfectly average, while
                       one zone under it runs away and dominates the episode.
                       This drives the loop from the worst zone instead.

  PIDEconomizerPolicy  Modulates recirculation rather than pinning it at the
                       ventilation cap. Cooling with return air while the
                       outside air is colder than the space is paid-for cooling
                       that the weather was offering free. The mix is
                       proportional: it blends outside and return air to land
                       on the supply air target, so very cold outside air
                       results in less of it being drawn, not none.

  PIDProportionalReheatPolicy
                       Modulates the reheat valve rather than switching it.

  PIDWaterResetPolicy  Resets the hot water supply temperature against demand,
                       so the loop is not held at its maximum whenever any
                       single zone calls for heat.

One further refinement runs through the chain: the comfort band is read from
each zone's own setpoint sensors instead of being rebuilt from the hour of day.
The reward measures deviation from those sensors, so where a reconstructed band
disagrees with them the controller either conditions a zone the reward already
considers comfortable, which is cost for no benefit, or neglects one it does
not. Reading the sensors makes the controller and the scorer agree on where the
edge is.
"""




# Native ranges from the action normalizer in every shipped config.
SUPPLY_AIR_MIN_K = 285.0
SUPPLY_AIR_MAX_K = 305.0
STATIC_PRESSURE_MAX_PA = 2000.0
# The simulator holds this floor while a handler runs, so a lower command
# still costs this much fan.
MIN_STATIC_PRESSURE_PA = 200.0

# Kelvin of outside-air advantage needed per pascal of fan static pressure
# before drawing outside air is cheaper than recirculating:
#
#     COP / (fan_efficiency * air_density * air_heat_capacity)
#   = 3.5 / (0.9 * 1.2 * 1006)  =  0.00322 K/Pa
#
# So 0.6 K at the 200 Pa floor and 6.4 K at full 2000 Pa fan. This scales with
# the cooling COP: a lower COP makes the chiller saving larger relative to the
# fan cost and lowers the bar for economizing, so the constant must be kept in
# step with AirHandler's cooling_coefficient_of_performance.
ECON_K_PER_PA = 0.00322


class PIDWorstZonePolicy(PIDPolicy):
    """`pid`, but the AHU loop tracks its worst zone rather than the average.

    Comfort shortfall is linear to 5 K outside the band and quadratic and
    unbounded past it, and it is summed over zones. One zone 10 K out costs far
    more than ten zones 1 K out. Averaging is the one aggregation that hides
    exactly that, and an air handler serving seven zones can read as perfectly
    in band while one of them is the whole episode's loss.
    """

    USE_SENSOR_BAND = False

    @property
    def name(self) -> str:
        return "pid_worst"

    def _zone_band(self, obs: np.ndarray, vav_id: str, low: float, high: float):
        """This zone's band. Falls back to the schedule band when unavailable."""
        if not self.USE_SENSOR_BAND:
            return low, high
        idx = self._sensor_band_indices.get(vav_id)
        if idx is None:
            return low, high
        lo_idx, hi_idx = idx
        lo = float(obs[lo_idx]) if lo_idx is not None else low
        hi = float(obs[hi_idx]) if hi_idx is not None else high
        # A band that came back inverted or empty is not usable; a controller
        # that trusted it would drive both ways at once.
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            return low, high
        return lo, hi

    def _build_sensor_band_indices(self) -> None:
        """Index of each zone's own heating and cooling setpoint sensor.

        These are raw kelvin, unlike the zone temperature, which is normalized.
        """
        self._sensor_band_indices: dict[str, tuple[int | None, int | None]] = {}
        names = list(self.field_names)

        def find(field: str):
            try:
                return names.index(field)
            except ValueError:
                return None

        for ctrl in self.ahu_controllers:
            for vav_id, _, _ in ctrl["zones"]:
                self._sensor_band_indices[vav_id] = (
                    find(f"{vav_id}_zone_air_heating_temperature_setpoint_sensor"),
                    find(f"{vav_id}_zone_air_cooling_temperature_setpoint_sensor"),
                )

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._build_sensor_band_indices()
        self._attach_recirculation_and_weather()

    def _attach_recirculation_and_weather(self) -> None:
        """`pid` builds no recirculation or outside-air entry; add them.

        `pid` predates the recirculation action and writes the cap to every
        recirculation index in one loop at the end, so its per-AHU map never
        needed either index. An economizer is a per-AHU decision and needs
        both.
        """
        names = list(self.field_names)
        for i, ctrl in enumerate(self.ahu_controllers):
            ctrl["recirc_idx"] = (
                self.recirculation_indices[i]
                if i < len(self.recirculation_indices)
                else None
            )
            field = f"ahs_{ctrl['ahu_id']}_outside_air_temperature_sensor"
            ctrl["oat_idx"] = names.index(field) if field in names else None

    def _worst_zone_error(self, obs: np.ndarray, ctrl, low: float, high: float) -> float:
        """Signed deadband error of the zone furthest outside its band.

        Sign is kept, so the AHU still knows which way to drive. When some
        zones are cold and others hot the largest magnitude wins, because that
        is the one the unbounded penalty is charging for.
        """
        worst = 0.0
        for vav_id, obs_idx, _ in ctrl["zones"]:
            lo, hi = self._zone_band(obs, vav_id, low, high)
            err = self._deadband_error(
                self._denormalize_zone_temp(obs[obs_idx]), lo, hi
            )
            if abs(err) > abs(worst):
                worst = err
        return worst

    def _command_zones(self, continuous, zones, obs, low, high) -> None:
        """Same as `pid`, except each zone is judged against its own band."""
        if not self.USE_SENSOR_BAND:
            return super()._command_zones(continuous, zones, obs, low, high)
        for vav_id, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            lo, hi = self._zone_band(obs, vav_id, low, high)
            zone_error = self._deadband_error(
                self._denormalize_zone_temp(obs[obs_idx]), lo, hi
            )
            damper = min(1.0, abs(zone_error) / self.zone_damper_full_k)
            damper_idx = action_indices.get("supply_air_damper_percentage_command")
            if damper_idx is not None:
                continuous[damper_idx] = 2.0 * damper - 1.0
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is not None:
                continuous[reheat_idx] = 1.0 if zone_error > 0 else -1.0

    # `pid.get_action` reduces the AHU's zones with np.mean. Rather than copy
    # the whole method to change one line, override the reduction it uses.
    def _ahu_error(self, obs: np.ndarray, ctrl, low: float, high: float) -> float:
        return self._worst_zone_error(obs, ctrl, low, high)

    def get_action(self, obs: np.ndarray) -> dict[str, Any]:
        hour, day = self._get_hour_and_day(obs)
        is_weekday = day < 5
        is_work_hour = self.morning_start_hour <= hour < self.evening_start_hour
        if is_weekday and is_work_hour:
            low_temp, high_temp = self.comfort_temp_window
        else:
            low_temp, high_temp = self.eco_temp_window

        continuous = np.zeros(self.n_continuous, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        any_heating = False
        any_active = False
        self._last_efforts: list[float] = []

        # Default every handler to the ventilation cap, then let the loop below
        # overwrite the ones it decides about. Testing for a leftover 0.0 after
        # the fact would misread a legitimate 50% recirculation command.
        for idx in self.recirculation_indices:
            continuous[idx] = 1.0

        for i, ctrl in enumerate(self.ahu_controllers):
            if not ctrl["zone_obs_indices"]:
                continue

            self._command_zones(continuous, ctrl["zones"], obs, low_temp, high_temp)
            error = self._ahu_error(obs, ctrl, low_temp, high_temp)

            state = self.pid_state[i]
            if error == 0.0:
                state["integral"] *= 0.5
                state["prev_error"] = 0.0
                continuous[ctrl["temp_idx"]] = -1.0
                if ctrl["pressure_idx"] is not None:
                    continuous[ctrl["pressure_idx"]] = -1.0
                if ctrl.get("recirc_idx") is not None:
                    continuous[ctrl["recirc_idx"]] = 1.0
                discrete[ctrl["run_discrete_idx"]] = 0
                continue

            state["integral"] = float(np.clip(
                state["integral"] + error, -self.integral_limit, self.integral_limit))
            derivative = error - state["prev_error"]
            state["prev_error"] = error

            u = float(np.clip(
                self.kp * error + self.ki * state["integral"] + self.kd * derivative,
                -1.0, 1.0))

            continuous[ctrl["temp_idx"]] = u
            # Recorded for the hot water reset in PIDWaterResetPolicy, which
            # needs to know whether the air side has any authority left. Kept
            # here rather than recomputed, because a reset that re-derived the
            # effort could drift out of step with what was actually commanded.
            self._last_efforts.append(u)
            if ctrl["pressure_idx"] is not None:
                continuous[ctrl["pressure_idx"]] = float(
                    np.clip(2.0 * abs(u) - 1.0, -1.0, 1.0))
            discrete[ctrl["run_discrete_idx"]] = 1

            if ctrl.get("recirc_idx") is not None:
                continuous[ctrl["recirc_idx"]] = self._recirculation(
                    obs, ctrl, u, low_temp, high_temp)

            any_active = True
            if error > 0:
                any_heating = True

        if self.water_temp_idx is not None:
            continuous[self.water_temp_idx] = 1.0 if any_heating else -1.0
        if self.water_pressure_idx is not None:
            continuous[self.water_pressure_idx] = 1.0 if any_active else -1.0
        if 0 <= self.water_discrete_idx < self.n_discrete:
            discrete[self.water_discrete_idx] = 1 if any_active else 0

        return {"continuous_action": continuous, "discrete_action": discrete}

    def _recirculation(self, obs, ctrl, u: float, low: float, high: float) -> float:
        """Hold the cap. Subclasses that economize override this."""
        return 1.0


class PIDEconomizerPolicy(PIDWorstZonePolicy):
    """`pid` plus a proportional economizer. Average-zone loop, as `pid` has.

    When the handler is cooling and the outside air is colder than the space,
    some of that cooling is available for the price of moving the air. The
    mixed air temperature is

        mixed = f * outside + (1 - f) * inside

    for an outside air fraction `f`, so the `f` that lands exactly on the
    supply air target is

        f = (inside - target) / (inside - outside)

    and the recirculation command is `1 - f`. Solving for the target rather
    than switching to full outside air is the whole point: in cold weather this
    asks for a *small* amount of outside air, where an on/off economizer has to
    choose between none and a blast of -30 C air.

    The simulator clamps recirculation at 70%, so the outside air fraction
    never falls below 30% whatever this returns. That is the ventilation floor
    and it is not something a policy can trade away.
    """

    USE_WORST_ZONE = False

    @property
    def name(self) -> str:
        return "pid_econ"

    def _ahu_error(self, obs: np.ndarray, ctrl, low: float, high: float) -> float:
        if self.USE_WORST_ZONE:
            return self._worst_zone_error(obs, ctrl, low, high)
        temps = [self._denormalize_zone_temp(obs[j]) for j in ctrl["zone_obs_indices"]]
        return self._deadband_error(float(np.mean(temps)), low, high)

    def _recirculation(self, obs, ctrl, u: float, low: float, high: float) -> float:
        oat_idx = ctrl.get("oat_idx")
        # Heating, or no weather reading: hold the heat in. Recirculated air is
        # the warmest air available and the handler cannot add heat itself.
        if u >= 0.0 or oat_idx is None:
            return 1.0

        outside = float(obs[oat_idx])
        inside = float(np.mean(
            [self._denormalize_zone_temp(obs[j]) for j in ctrl["zone_obs_indices"]]))
        if not np.isfinite(outside) or outside >= inside:
            return 1.0

        # Free cooling is not free. The exhaust fan moves
        # `air_flow_rate * (1 - recirculation)`, so every extra unit of outside
        # air costs fan power even though it saves chiller work. Per unit of
        # outside air fraction, and per unit of flow:
        #
        #     chiller saved = rho * cp * (return - outside) / COP
        #     exhaust paid  = static_pressure / fan_efficiency
        #
        # so it only pays once
        #
        #     (return - outside) > static_pressure * COP / (fan_eff * rho * cp)
        #
        # Below this threshold the extra fan work required to move outside
        # air costs more than the chiller saving it buys, so economizing on a
        # small temperature advantage is a net loss.
        pressure_pa = (self._commanded_pressure(u) + 1.0) * 0.5 * STATIC_PRESSURE_MAX_PA
        pressure_pa = max(pressure_pa, MIN_STATIC_PRESSURE_PA)
        breakeven_k = pressure_pa * ECON_K_PER_PA
        if (inside - outside) <= breakeven_k:
            return 1.0

        target = SUPPLY_AIR_MIN_K + (u + 1.0) * 0.5 * (SUPPLY_AIR_MAX_K - SUPPLY_AIR_MIN_K)
        # Never mix below the band's lower edge. Overshooting into heating
        # would trade a cooling bill for a larger reheat bill.
        target = max(target, low)
        if inside <= target:
            return 1.0

        f = (inside - target) / (inside - outside)
        recirculation = float(np.clip(1.0 - f, 0.0, 1.0))
        return 2.0 * recirculation - 1.0

    @staticmethod
    def _commanded_pressure(u: float) -> float:
        """The normalized pressure this controller is about to write."""
        return float(np.clip(2.0 * abs(u) - 1.0, -1.0, 1.0))


class PIDProportionalReheatPolicy(PIDEconomizerPolicy):
    """`pid` with a modulating reheat valve instead of an on/off one.

    Every baseline in this repo drives the valve binary:

        continuous[reheat_idx] = 1.0 if zone_error > 0 else -1.0

    but `reheat_valve_percentage_command` is a continuous 0-1 position. Fully
    opening it for a zone that is 0.3 K cold delivers the same heat as for one
    that is 8 K cold, so the near-miss zone is driven straight through its band
    and out the other side.

    An on/off valve on a slow plant therefore spends more zone-time above the
    band than below it, with the damper still open through most of the
    overshoot: the controller ends up net-heating zones it was trying to bring
    just into range. This is a resolution problem rather than a lack of
    authority -- a single zone's own actuators can move it many kelvin.

    The valve is opened in proportion to the shortfall, full open at
    REHEAT_FULL_K below the heating setpoint, and shut once the zone is inside
    its band.
    """

    # Shortfall in K at which the valve is fully open. Matched to the AHU
    # loop's own 1/kp = 2 K so the two layers reach saturation together.
    REHEAT_FULL_K = 2.0

    @property
    def name(self) -> str:
        return "pid_prop"

    def _command_zones(self, continuous, zones, obs, low, high) -> None:
        for vav_id, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            lo, hi = self._zone_band(obs, vav_id, low, high)
            zone_error = self._deadband_error(
                self._denormalize_zone_temp(obs[obs_idx]), lo, hi
            )
            damper = min(1.0, abs(zone_error) / self.zone_damper_full_k)
            damper_idx = action_indices.get("supply_air_damper_percentage_command")
            if damper_idx is not None:
                continuous[damper_idx] = 2.0 * damper - 1.0
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is not None:
                # zone_error > 0 means below the heating setpoint.
                opening = min(1.0, max(0.0, zone_error) / self.REHEAT_FULL_K)
                continuous[reheat_idx] = 2.0 * opening - 1.0


class PIDWaterResetPolicy(PIDProportionalReheatPolicy):
    """Proportional reheat valve AND the hot water temperature reset on demand.

    Both halves are needed, and each is weak alone.

    The hot water loop is building-wide. A demand reset on its own is therefore
    set by the coldest zone in the building: one zone several kelvin below its
    band drives the loop to its maximum, and every satisfied zone still sitting
    at its damper floor receives that very hot water anyway. A proportional
    valve on its own helps, but it is throttling air that is still far hotter
    than the zone needs.

    Together the loop runs only as hot as the worst zone requires, and each
    valve opens only as far as its own zone requires.

    WHY A RESET RATHER THAN A FIXED TEMPERATURE

    Pinning the loop low is not a solution: a hard winter climate genuinely
    needs the full water temperature. So the loop is reset against the demand
    that actually exists, linearly, saturating at RESET_FULL_K below the band.

    PROPORTIONAL DROOP, AND THE SATURATION TERM

    A purely proportional reset has a steady-state offset, and here that offset
    is the whole error: the loop only commands full water when the mean zone is
    already RESET_FULL_K short. In a climate that genuinely needs full water the
    controller therefore settles on a building that is RESET_FULL_K cold, which
    is precisely where it must not be.

    An integrator is the textbook remedy and is the wrong one here. Raising the
    water temperature satisfies the bulk of the zones sooner, so the air handler
    idles sooner, so an outlier zone loses its airflow and drifts further. Since
    the comfort penalty is unbounded in the worst zone, widening the spread at
    both ends is the worst available outcome. The mechanism is left in the class
    with its gain defaulting to zero, so the next reader does not rebuild it.

    Outdoor-air reset is the other textbook answer. It works, but the crossover
    temperature it needs differs sharply between cold climates, and fitting that
    ramp to the scenarios in hand would not transfer to unseen buildings.

    Instead the reset takes the greater of two signals:

      demand      how far the mean zone is below its band
      saturation  how hard the air side is already working

    The second answers a different question -- not "is the building cold" but
    "has the air side run out of authority". In a severe winter every handler
    sits at full effort all episode, the air side has nothing left to give, and
    the only remaining lever is hotter water. In a mild climate effort is near
    zero for most of the day, so the loop stays cool and follows only the
    morning ramp. Reading saturation off the plant itself is what makes this
    transfer, where a fitted outdoor-air curve would not.
    """

    # Proportional band: mean zone shortfall, in K, that alone commands full
    # water.
    RESET_FULL_K = 4.0
    # Weight on the air-side saturation term. See `_water_fraction`.
    RESET_EFFORT_WEIGHT = 1.0
    # Which handler to believe when they disagree, as a quantile over the
    # per-handler heating efforts. 0.0 is the least loaded handler, 1.0 the most.
    #
    # The quantile matters because handlers disagree. A low quantile asks
    # whether the least loaded handler is saturated, a high one whether the
    # most loaded is.
    #
    # The mean and the median both lose ground in shoulder weather, because a
    # single unreachable cold zone can pin its own handler at full effort for a
    # whole episode and drag the summary up with it. The minimum is too strict
    # in the other direction: one idle handler in a wide building vetoes the
    # loop for every other handler. The lower quartile ignores a lone saturated
    # handler without letting a lone idle one veto, which is why it is the
    # default.
    RESET_EFFORT_QUANTILE = 0.25
    # Integral gain and decay. Both left in place, and the gain defaults to
    # zero, because the integrator was built, measured and rejected; see the
    # class docstring. Keeping the mechanism costs three lines and stops the
    # next reader from rebuilding it.
    RESET_DEADBAND_K = 0.25
    RESET_INTEGRAL_GAIN = 0.0
    RESET_INTEGRAL_DECAY = 0.99

    @property
    def name(self) -> str:
        return "pid_water"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._water_integral = 0.0

    def _heating_shortfall(self, obs, low: float, high: float) -> float:
        """Building-wide heating demand, as the MEAN zone shortfall.

        Deliberately not the coldest zone. The hot water loop is building-wide,
        so letting a single outlier set its temperature drives the loop to its
        maximum and delivers that water to every satisfied zone sitting at its
        damper floor.

        The outlier is also frequently unreachable: a zone starved of airflow
        stays cold whatever the loop temperature, so heat spent chasing it buys
        nothing while damaging every other zone. The mean tracks the demand the
        loop can actually serve, and it remains the better statistic in cold
        climates where the heat is genuinely needed.
        """
        shortfalls = []
        for ctrl in self.ahu_controllers:
            for vav_id, obs_idx, _ in ctrl["zones"]:
                lo, _hi = self._zone_band(obs, vav_id, low, high)
                shortfalls.append(
                    max(0.0, lo - self._denormalize_zone_temp(obs[obs_idx]))
                )
        return float(np.mean(shortfalls)) if shortfalls else 0.0

    def get_action(self, obs):
        action = super().get_action(obs)
        if self.water_temp_idx is None:
            return action

        hour, day = self._get_hour_and_day(obs)
        is_work = day < 5 and self.morning_start_hour <= hour < self.evening_start_hour
        low, high = self.comfort_temp_window if is_work else self.eco_temp_window
        shortfall = self._heating_shortfall(obs, low, high)

        # The integrator is updated on every step, including steps where no
        # handler is calling for heat. Updating it only while heating would let
        # it hold a stale wound-up value across a whole warm afternoon and then
        # dump full water into the first zone that asks at dusk.
        if shortfall > self.RESET_DEADBAND_K:
            self._water_integral += self.RESET_INTEGRAL_GAIN * (
                shortfall - self.RESET_DEADBAND_K)
        else:
            self._water_integral *= self.RESET_INTEGRAL_DECAY
        self._water_integral = float(np.clip(self._water_integral, 0.0, 1.0))

        cont = action["continuous_action"]
        if cont[self.water_temp_idx] <= 0.0:
            return action  # not heating; leave the loop cold

        frac = self._water_fraction(shortfall)
        cont[self.water_temp_idx] = np.float32(2.0 * frac - 1.0)
        return action

    def _water_fraction(self, shortfall: float) -> float:
        """How hot to run the loop, as a fraction of the 310-350 K range.

        Two terms, and the larger wins.

        The DEMAND term is the mean zone shortfall over the proportional band,
        as described above.

        The SATURATION term is the mean heating effort the AHU loop is already
        commanding. It answers a different question: not "how cold is the
        building" but "has the air side run out of authority". Those come apart
        exactly where the demand term fails. In deep cold every handler sits at
        full effort all episode, so the air side has nothing left to give and
        the only remaining lever is hotter water. In shoulder weather effort is
        near zero most of the day, so the loop stays cool and only follows the
        morning ramp.

        Why this and not an integrator, and not outdoor-air reset:

        * An integrator removes the droop in cold weather but widens the zone
          spread at both ends. Hotter water satisfies the bulk of the zones
          sooner, so the handler idles sooner, so an outlier zone loses its
          airflow and drifts further. Since the comfort penalty is unbounded in
          the worst zone, that trade is a loss.
        * Outdoor-air reset is the textbook answer and does work, but the
          crossover temperature it needs differs sharply between cold climates.
          Fitting that ramp to the scenarios in hand would not transfer to
          unseen buildings. The saturation term reads the same distinction off
          the plant itself.

        Note the saturation term also pays for itself on energy, which was not
        the reason for adding it. In the coldest scenarios, cool water costs
        MORE than hot: the coil cannot deliver, so the fan sits at full static
        pressure all episode moving air that is not warm enough. Per unit of
        heat delivered, hotter water trades fan electricity for boiler gas, and
        electricity is the dearer of the two in both price and carbon.
        """
        demand = shortfall / self.RESET_FULL_K
        efforts = getattr(self, "_last_efforts", None)
        saturation = 0.0
        if efforts and self.RESET_EFFORT_WEIGHT:
            heating = [max(0.0, u) for u in efforts]
            saturation = self.RESET_EFFORT_WEIGHT * float(
                np.quantile(heating, self.RESET_EFFORT_QUANTILE))
        return float(np.clip(max(demand, saturation) + self._water_integral,
                             0.0, 1.0))

"""Pluggable hourly Outside Air Temperature (OAT) forecasters."""




STEPS_PER_HOUR = 12  # 5-minute timestep


@dataclass
class ForecastResult:
    """Multi-hour ahead OAT forecast in Kelvin."""

    p50_k: np.ndarray
    p10_k: np.ndarray
    p90_k: np.ndarray


class BaseOATForecaster:
    """Base interface for hourly OAT forecasters."""

    arm_name: str = "base"

    def __init__(self, horizon_hours: int = 6) -> None:
        self.horizon_hours = horizon_hours
        self.step_count = 0
        self.forecast_calls = 0
        self.forecast_wall_s = 0.0
        self._cached: ForecastResult | None = None
        self._oat_history_5m: deque[float] = deque(maxlen=24 * 7 * STEPS_PER_HOUR)

    def reset(self) -> None:
        self.step_count = 0
        self.forecast_calls = 0
        self.forecast_wall_s = 0.0
        self._cached = None
        self._oat_history_5m.clear()

    def update_and_forecast(self, oat_k: float) -> ForecastResult | None:
        if np.isfinite(oat_k):
            self._oat_history_5m.append(float(oat_k))
        should_refresh = (self.step_count % STEPS_PER_HOUR == 0)
        self.step_count += 1
        if not should_refresh:
            return self._cached

        t0 = time.perf_counter()
        self._cached = self._predict_hourly(step_idx=self.step_count - 1)
        self.forecast_wall_s += time.perf_counter() - t0
        if self._cached is not None:
            self.forecast_calls += 1
        return self._cached

    def _hourly_series(self) -> np.ndarray:
        arr = np.asarray(self._oat_history_5m, dtype=np.float32)
        n_full = len(arr) // STEPS_PER_HOUR
        if n_full == 0:
            return arr[-1:] if len(arr) else np.array([], dtype=np.float32)
        return arr[-n_full * STEPS_PER_HOUR :].reshape(n_full, STEPS_PER_HOUR).mean(axis=1)

    def _predict_hourly(self, step_idx: int) -> ForecastResult | None:
        raise NotImplementedError


class NoForecast(BaseOATForecaster):
    """Arm A: No forecast (pure reactive control)."""

    arm_name = "Arm_A_Baseline"

    def _predict_hourly(self, step_idx: int) -> ForecastResult | None:
        return None


class PeriodicHeuristicForecaster(BaseOATForecaster):
    """Arm B: 24h-lag + damped 1h linear trend forecaster (pure NumPy)."""

    arm_name = "Arm_B_Heuristic"

    def _predict_hourly(self, step_idx: int) -> ForecastResult | None:
        hourly = self._hourly_series()
        if len(hourly) < 2:
            return None
        curr = float(hourly[-1])
        slope = float(np.clip(hourly[-1] - hourly[-2], -3.0, 3.0))
        h = np.arange(1, self.horizon_hours + 1, dtype=np.float32)

        if len(hourly) >= 24 + self.horizon_hours:
            yesterday_now = float(hourly[-25])
            yesterday_future = hourly[-24 : -24 + self.horizon_hours]
            p50 = yesterday_future + (curr - yesterday_now) * np.exp(-0.15 * h)
        else:
            p50 = curr + slope * (1.0 - np.exp(-0.35 * h)) / 0.35

        spread = 0.8 + 0.35 * h
        return ForecastResult(p50_k=p50, p10_k=p50 - spread, p90_k=p50 + spread)

"""Generalized, parameterizable HVAC policy built on starter-kit pid_water."""






@dataclass
class PolicyConfig:
    """Tunable parameters for GeneralizedHVACPolicy (serializable to JSON)."""

    kp: float = 0.50
    ki: float = 0.02
    kd: float = 0.10
    zone_damper_full_k: float = 2.0
    reheat_full_k: float = 2.0
    reset_full_k: float = 3.2
    reset_effort_quantile: float = 0.40
    worst_zone_guard_k: float = 3.0
    use_sensor_band: bool = True
    fix_hws_cooling_bug: bool = True
    precool_trigger_delta_k: float = 6.0
    precool_margin_k: float = 0.5
    preheat_trigger_delta_k: float = 15.0
    preheat_margin_k: float = 0.5
    pre_window_hours: int = 2
    fix_sensor_band_precond: bool = True
    fix_hws_pump_demand_check: bool = True
    worst_zone_guard_heat_k: float | None = None
    enable_cold_hws_boost: bool = False
    cold_oat_hws_floor_start_k: float = 273.15
    cold_oat_hws_floor_full_k: float = 253.15
    use_continuous_precond: bool = False
    precool_ramp_start_delta_k: float = -1.0
    preheat_ramp_start_delta_k: float = 5.0
    adaptive_eco_cap: bool = False
    eco_cap_hot_oat_margin_k: float = 2.0
    eco_cap_cold_oat_k: float = 263.15
    use_soft_heat_guard: bool = False
    soft_heat_guard_deadband_k: float = 0.0
    soft_heat_guard_max_oat_delta_k: float | None = None
    soft_heat_guard_min_w: float = 0.35
    soft_heat_guard_ramp_lo_k: float = 2.0
    soft_heat_guard_ramp_hi_k: float = 4.5
    use_heat_pressure_scaling: bool = False
    heat_pressure_min_scale: float = 0.35
    use_continuous_eco_cap: bool = False
    cold_temper_margin_k: float = 0.0

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def load(cls, path: Path) -> "PolicyConfig":
        return cls(**json.loads(path.read_text()))


class GeneralizedHVACPolicy(PIDWaterResetPolicy):
    """Generalized HVAC controller with safety guards, HWS fix, and residual hook."""

    def __init__(
        self,
        building_meta_data: Mapping[str, Any],
        action_spec: Any,
        config: PolicyConfig | None = None,
        forecaster: BaseOATForecaster | None = None,
        residual_fn: Callable[[np.ndarray, dict[str, np.ndarray], Mapping[str, Any]], np.ndarray] | None = None,
    ) -> None:
        self.cfg = config or PolicyConfig()
        self.USE_SENSOR_BAND = bool(self.cfg.use_sensor_band)
        self.REHEAT_FULL_K = float(self.cfg.reheat_full_k)
        self.RESET_FULL_K = float(self.cfg.reset_full_k)
        self.RESET_EFFORT_QUANTILE = float(self.cfg.reset_effort_quantile)

        super().__init__(
            building_meta_data,
            action_spec,
            kp=self.cfg.kp,
            ki=self.cfg.ki,
            kd=self.cfg.kd,
            zone_damper_full_k=self.cfg.zone_damper_full_k,
        )
        self.forecaster = forecaster or NoForecast()
        self.residual_fn = residual_fn

        self._primary_oat_idx = next(
            (c["oat_idx"] for c in self.ahu_controllers if c.get("oat_idx") is not None),
            None,
        )
        names = list(self.field_names)
        self._comfort_soon_idx = next(
            (i for i, n in enumerate(names) if "comfort" in n and "soon" in n),
            None,
        )
        self._reheat_action_indices = [
            action_indices["reheat_valve_percentage_command"]
            for ctrl in self.ahu_controllers
            for _, _, action_indices in ctrl["zones"]
            if "reheat_valve_percentage_command" in action_indices
        ]
        self._active_band_override: tuple[float, float] | None = None
        self._last_oat_k: float = float("nan")
        self._any_ahu_heating: bool = False

    def _zone_band(self, obs: np.ndarray, vav_id: str, low: float, high: float) -> tuple[float, float]:
        lo, hi = super()._zone_band(obs, vav_id, low, high)
        if self.cfg.fix_sensor_band_precond and self._active_band_override is not None:
            ov_lo, ov_hi = self._active_band_override
            lo = max(lo, ov_lo)
            hi = min(hi, ov_hi)
            if hi <= lo + 0.1:
                hi = lo + 0.1
        return lo, hi

    def _command_zones(
        self,
        continuous: np.ndarray,
        zones: list[tuple[str, int, dict[str, int]]],
        obs: np.ndarray,
        low: float,
        high: float,
    ) -> None:
        super()._command_zones(continuous, zones, obs, low, high)
        if self.cfg.cold_temper_margin_k <= 0.0 or not np.isfinite(self._last_oat_k) or self._last_oat_k >= 273.15:
            return
        for vav_id, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is None:
                continue
            lo, hi = self._zone_band(obs, vav_id, low, high)
            temp = self._denormalize_zone_temp(obs[obs_idx])
            if self._deadband_error(temp, lo, hi) == 0.0:
                margin = lo + self.cfg.cold_temper_margin_k - temp
                if margin > 0.0:
                    opening = min(0.35, margin / self.REHEAT_FULL_K)
                    continuous[reheat_idx] = max(float(continuous[reheat_idx]), 2.0 * opening - 1.0)

    def _ahu_error(self, obs: np.ndarray, ctrl: dict[str, Any], low: float, high: float) -> float:
        if self.cfg.use_soft_heat_guard:
            temps = []
            errs = []
            for vav_id, obs_idx, _ in ctrl["zones"]:
                temp = self._denormalize_zone_temp(obs[obs_idx])
                temps.append(temp)
                lo, hi = self._zone_band(obs, vav_id, low, high)
                errs.append(self._deadband_error(temp, lo, hi))
            mean_err = self._deadband_error(float(np.mean(temps)), low, high)
            worst_cool = min(0.0, min(errs))
            worst_heat = max(0.0, max(errs))
            cold_frac = sum(1 for e in errs if e > 0.0) / max(1, len(errs))
            comfort_lo = self.comfort_temp_window[0]
            oat_allows_heat_guard = (
                self.cfg.soft_heat_guard_max_oat_delta_k is None
                or not np.isfinite(self._last_oat_k)
                or self._last_oat_k < comfort_lo - self.cfg.soft_heat_guard_max_oat_delta_k
                or worst_heat >= self.cfg.soft_heat_guard_ramp_hi_k
            )
            if abs(worst_cool) >= self.cfg.worst_zone_guard_k or (
                abs(worst_cool) > worst_heat and worst_heat < self.cfg.soft_heat_guard_ramp_hi_k
            ):
                err = worst_cool if abs(worst_cool) >= self.cfg.worst_zone_guard_k else mean_err
            elif mean_err < 0.0 and worst_heat < self.cfg.soft_heat_guard_ramp_hi_k:
                err = mean_err
            elif oat_allows_heat_guard and worst_heat >= self.cfg.soft_heat_guard_deadband_k and worst_heat > 0.0:
                t = float(
                    np.clip(
                        (worst_heat - self.cfg.soft_heat_guard_ramp_lo_k)
                        / max(0.1, self.cfg.soft_heat_guard_ramp_hi_k - self.cfg.soft_heat_guard_ramp_lo_k),
                        0.0,
                        1.0,
                    )
                )
                w = self.cfg.soft_heat_guard_min_w + (1.0 - self.cfg.soft_heat_guard_min_w) * t
                err = max(max(0.0, mean_err), w * worst_heat)
            else:
                err = mean_err
            ctrl["_last_err"] = err
            ctrl["_last_worst_heat"] = worst_heat
            ctrl["_last_cold_frac"] = cold_frac
            if err > 0.0:
                self._any_ahu_heating = True
            return err

        worst = self._worst_zone_error(obs, ctrl, low, high)
        heat_guard = (
            self.cfg.worst_zone_guard_k
            if self.cfg.worst_zone_guard_heat_k is None
            else self.cfg.worst_zone_guard_heat_k
        )
        if (worst < 0.0 and abs(worst) >= self.cfg.worst_zone_guard_k) or (
            worst > 0.0 and worst >= heat_guard
        ):
            err = worst
        else:
            temps = [self._denormalize_zone_temp(obs[j]) for j in ctrl["zone_obs_indices"]]
            err = self._deadband_error(float(np.mean(temps)), low, high)
        ctrl["_last_err"] = err
        ctrl["_last_worst_heat"] = max(0.0, worst)
        ctrl["_last_cold_frac"] = 1.0
        if err > 0.0:
            self._any_ahu_heating = True
        return err

    def _cold_oat_hws_floor(self) -> float:
        if not self.cfg.enable_cold_hws_boost or not np.isfinite(self._last_oat_k):
            return 0.0
        start_k = self.cfg.cold_oat_hws_floor_start_k
        full_k = self.cfg.cold_oat_hws_floor_full_k
        if self._last_oat_k >= start_k:
            return 0.0
        return float(np.clip((start_k - self._last_oat_k) / max(1.0, start_k - full_k), 0.0, 1.0))

    def _water_fraction(self, shortfall: float) -> float:
        frac = super()._water_fraction(shortfall)
        return max(frac, self._cold_oat_hws_floor())

    def _hours_to_morning_start(self, hour: int, day: int) -> int:
        if day < 5 and hour < self.morning_start_hour:
            return self.morning_start_hour - hour
        if ((day + 1) % 7) < 5 and hour >= self.evening_start_hour:
            return (24 - hour) + self.morning_start_hour
        return 999

    def _forecast_band_shift(
        self,
        obs: np.ndarray,
        hour: int,
        day: int,
        low: float,
        high: float,
        forecast: ForecastResult | None,
        *,
        is_work: bool | None = None,
        oat_k: float = float("nan"),
    ) -> tuple[float, float]:
        comfort_lo, comfort_hi = self.comfort_temp_window
        comfort_soon = (
            self._comfort_soon_idx is not None and float(obs[self._comfort_soon_idx]) > 0.5
        )
        if not self.cfg.use_continuous_precond:
            if forecast is None or day >= 5:
                return low, high
            hours_to_start = self.morning_start_hour - hour
            if not (comfort_soon or (0 < hours_to_start <= self.cfg.pre_window_hours)):
                return low, high

            adj_low, adj_high = low, high
            if float(np.max(forecast.p90_k)) > comfort_hi + self.cfg.precool_trigger_delta_k:
                adj_high = min(adj_high, comfort_hi - self.cfg.precool_margin_k)
            if float(np.min(forecast.p10_k)) < comfort_lo - self.cfg.preheat_trigger_delta_k:
                adj_low = max(adj_low, comfort_lo + self.cfg.preheat_margin_k)
            return adj_low, adj_high

        if is_work is None:
            is_work = day < 5 and self.morning_start_hour <= hour < self.evening_start_hour

        adj_low, adj_high = low, high
        if self.cfg.use_continuous_eco_cap and not is_work:
            is_weekend_day = (day >= 5) and not (day == 6 and hour >= self.evening_start_hour)
            hot_vals = [
                v
                for v in (oat_k, float(np.max(forecast.p50_k)) if forecast is not None else float("nan"))
                if np.isfinite(v)
            ]
            if hot_vals:
                hot_ref = max(hot_vals)
                if hot_ref > comfort_hi:
                    hot_sev = float(np.clip((hot_ref - comfort_hi) / 10.0, 0.0, 1.0))
                    cap_hi = comfort_hi + (2.0 - 1.5 * hot_sev) + (1.0 if is_weekend_day else 0.0)
                    adj_high = min(adj_high, cap_hi)

            cold_vals = [
                v
                for v in (oat_k, float(np.min(forecast.p50_k)) if forecast is not None else float("nan"))
                if np.isfinite(v)
            ]
            if cold_vals:
                cold_ref = min(cold_vals)
                if cold_ref < comfort_lo - 15.0:
                    cold_sev = float(np.clip(((comfort_lo - 15.0) - cold_ref) / 35.0, 0.0, 1.0))
                    cap_lo = comfort_lo - (2.8 - 2.6 * cold_sev) - (
                        1.5 if (is_weekend_day and cold_ref >= 248.15) else 0.0
                    )
                    adj_low = max(adj_low, cap_lo)
        elif self.cfg.adaptive_eco_cap and not is_work:
            hot_ref = float(np.max(forecast.p50_k)) if forecast is not None else float("nan")
            if (np.isfinite(oat_k) and oat_k > comfort_hi - self.cfg.eco_cap_hot_oat_margin_k) or (
                np.isfinite(hot_ref) and hot_ref > comfort_hi - self.cfg.eco_cap_hot_oat_margin_k
            ):
                adj_high = min(adj_high, comfort_hi + 1.5)
            cold_ref = float(np.min(forecast.p50_k)) if forecast is not None else float("nan")
            if (np.isfinite(oat_k) and oat_k < self.cfg.eco_cap_cold_oat_k) or (
                np.isfinite(cold_ref) and cold_ref < self.cfg.eco_cap_cold_oat_k
            ):
                adj_low = max(adj_low, comfort_lo - 1.5)

        hours_to_start = self._hours_to_morning_start(hour, day)
        in_weekday_pre = (day < 5 and comfort_soon and not is_work) or (
            0 < hours_to_start <= self.cfg.pre_window_hours
        )
        if in_weekday_pre and forecast is not None:
            prog = (
                1.0
                if comfort_soon
                else float(np.clip(1.0 - (hours_to_start - 1) / max(1, self.cfg.pre_window_hours), 0.25, 1.0))
            )
            precool_thresh = comfort_hi + self.cfg.precool_ramp_start_delta_k
            p90_max = float(np.max(forecast.p90_k))
            if p90_max > precool_thresh:
                sev = float(np.clip((p90_max - precool_thresh) / 10.0, 0.0, 1.0))
                adj_high = min(adj_high, comfort_hi - prog * sev * self.cfg.precool_margin_k)

            preheat_thresh = comfort_lo - self.cfg.preheat_ramp_start_delta_k
            p10_min = float(np.min(forecast.p10_k))
            if p10_min < preheat_thresh:
                sev = float(np.clip((preheat_thresh - p10_min) / 15.0, 0.0, 1.0))
                adj_low = max(adj_low, comfort_lo + prog * sev * self.cfg.preheat_margin_k)

        if adj_high <= adj_low + 0.5:
            mid = 0.5 * (comfort_lo + comfort_hi)
            adj_low = min(adj_low, mid - 0.25)
            adj_high = max(adj_high, adj_low + 0.5)
        return adj_low, adj_high

    def get_action(self, obs: np.ndarray) -> dict[str, np.ndarray]:
        oat_k = float(obs[self._primary_oat_idx]) if self._primary_oat_idx is not None else float("nan")
        self._last_oat_k = oat_k
        forecast = self.forecaster.update_and_forecast(oat_k)

        hour, day = self._get_hour_and_day(obs)
        is_work = day < 5 and self.morning_start_hour <= hour < self.evening_start_hour
        base_low, base_high = self.comfort_temp_window if is_work else self.eco_temp_window
        adj_low, adj_high = self._forecast_band_shift(
            obs, hour, day, base_low, base_high, forecast, is_work=is_work, oat_k=oat_k
        )

        orig_eco = self.eco_temp_window
        self._any_ahu_heating = False
        if not is_work:
            self.eco_temp_window = (adj_low, adj_high)
            if self.cfg.fix_sensor_band_precond and (adj_low > base_low or adj_high < base_high):
                self._active_band_override = (adj_low, adj_high)
        try:
            action = super().get_action(obs)
        finally:
            self.eco_temp_window = orig_eco
            self._active_band_override = None

        cont = np.asarray(action["continuous_action"], dtype=np.float32)
        disc = np.asarray(action["discrete_action"], dtype=np.int32)

        if self.cfg.use_heat_pressure_scaling:
            for ctrl in self.ahu_controllers:
                p_idx = ctrl.get("pressure_idx")
                if p_idx is None:
                    continue
                if disc[ctrl["run_discrete_idx"]] == 1 and ctrl.get("_last_err", 0.0) > 0.0:
                    sev = float(np.clip((ctrl.get("_last_worst_heat", 0.0) - 2.5) / 2.0, 0.0, 1.0))
                    scale = float(
                        np.clip(
                            max(self.cfg.heat_pressure_min_scale, ctrl.get("_last_cold_frac", 1.0), sev),
                            self.cfg.heat_pressure_min_scale,
                            1.0,
                        )
                    )
                    u_eff = abs(float(cont[ctrl["temp_idx"]])) * scale
                    cont[p_idx] = np.float32(np.clip(2.0 * u_eff - 1.0, -1.0, 1.0))

        if self.cfg.fix_hws_cooling_bug:
            if self.cfg.fix_hws_pump_demand_check:
                any_vav_reheat = any(cont[idx] > -0.99 for idx in self._reheat_action_indices)
                any_heating = self._any_ahu_heating or any_vav_reheat
                if any_heating:
                    if 0 <= self.water_discrete_idx < self.n_discrete:
                        disc[self.water_discrete_idx] = 1
                    if self.water_pressure_idx is not None:
                        cont[self.water_pressure_idx] = 1.0
                    if (
                        self.cfg.enable_cold_hws_boost
                        and self.water_temp_idx is not None
                        and cont[self.water_temp_idx] <= -0.99
                    ):
                        cold_floor = self._cold_oat_hws_floor()
                        if cold_floor > 0.0:
                            cont[self.water_temp_idx] = np.float32(2.0 * cold_floor - 1.0)
                else:
                    if self.water_temp_idx is not None:
                        cont[self.water_temp_idx] = -1.0
                    if self.water_pressure_idx is not None:
                        cont[self.water_pressure_idx] = -1.0
                    if 0 <= self.water_discrete_idx < self.n_discrete:
                        disc[self.water_discrete_idx] = 0
            else:
                is_heating = self.water_temp_idx is not None and cont[self.water_temp_idx] > -0.99
                if not is_heating:
                    if self.water_pressure_idx is not None:
                        cont[self.water_pressure_idx] = -1.0
                    if 0 <= self.water_discrete_idx < self.n_discrete:
                        disc[self.water_discrete_idx] = 0

        if self.residual_fn is not None:
            delta = np.asarray(self.residual_fn(obs, action, self.metadata), dtype=np.float32)
            if delta.shape == cont.shape and np.all(np.isfinite(delta)):
                cont = cont + delta

        cont = np.nan_to_num(cont, nan=-1.0, posinf=1.0, neginf=-1.0)
        cont = np.clip(cont, -1.0, 1.0).astype(np.float32)
        disc = np.clip(disc, 0, 1).astype(np.int32)
        return {"continuous_action": cont, "discrete_action": disc}

BEST_CONFIG = PolicyConfig(
    kp=0.5,
    ki=0.02,
    kd=0.1,
    zone_damper_full_k=2.0,
    reheat_full_k=1.5,
    reset_full_k=3.0,
    reset_effort_quantile=0.5,
    worst_zone_guard_k=3.0,
    use_sensor_band=True,
    fix_hws_cooling_bug=True,
    precool_trigger_delta_k=6.0,
    precool_margin_k=1.0,
    preheat_trigger_delta_k=15.0,
    preheat_margin_k=1.0,
    pre_window_hours=3,
    fix_sensor_band_precond=True,
    fix_hws_pump_demand_check=True,
    worst_zone_guard_heat_k=999.0,
    enable_cold_hws_boost=True,
    cold_oat_hws_floor_start_k=273.15,
    cold_oat_hws_floor_full_k=253.15,
    use_continuous_precond=True,
    precool_ramp_start_delta_k=-1.0,
    preheat_ramp_start_delta_k=5.0,
    adaptive_eco_cap=False,
    eco_cap_hot_oat_margin_k=2.0,
    eco_cap_cold_oat_k=263.15,
    use_soft_heat_guard=True,
    soft_heat_guard_deadband_k=1.5,
    soft_heat_guard_max_oat_delta_k=4.0,
    soft_heat_guard_min_w=0.35,
    soft_heat_guard_ramp_lo_k=2.0,
    soft_heat_guard_ramp_hi_k=4.5,
    use_heat_pressure_scaling=True,
    heat_pressure_min_scale=0.35,
    use_continuous_eco_cap=True,
    cold_temper_margin_k=0.35,
)


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Competition submission policy wrapping GeneralizedHVACPolicy."""

    def __init__(self, building_meta_data: Mapping[str, Any], action_spec: Any) -> None:
        super().__init__(building_meta_data, action_spec)
        fc = PeriodicHeuristicForecaster() if True else NoForecast()
        self._impl = GeneralizedHVACPolicy(
            building_meta_data, action_spec, config=BEST_CONFIG, forecaster=fc
        )
        self._cont_low = np.asarray(action_spec["continuous_action"].low, dtype=np.float32)
        self._cont_high = np.asarray(action_spec["continuous_action"].high, dtype=np.float32)
        self._disc_low = np.asarray(action_spec["discrete_action"].low, dtype=np.int32)
        self._disc_high = np.asarray(action_spec["discrete_action"].high, dtype=np.int32)

    def get_action(self, obs: np.ndarray) -> dict[str, np.ndarray]:
        action = self._impl.get_action(obs)
        cont = np.clip(
            np.nan_to_num(action["continuous_action"], nan=-1.0, posinf=1.0, neginf=-1.0),
            self._cont_low,
            self._cont_high,
        ).astype(np.float32)
        disc = np.clip(
            action["discrete_action"],
            self._disc_low,
            self._disc_high,
        ).astype(np.int32)
        return {"continuous_action": cont, "discrete_action": disc}


if __name__ == "__main__":
    smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run(
        competition_data_folder=COMPETITION_DATA_FOLDER,
    )
